In [1]:
from pathlib import Path
import json
import sys
import subprocess

REQ_FILE = Path("requirements.txt") if Path("requirements.txt").exists() else Path("requirement.txt")
if REQ_FILE.exists():
    try:
        import torch
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(REQ_FILE)])

PROJECT_ROOT = Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from food_cv.classifier import build_resnet50_classifier
from food_cv.config import ProjectPaths
from food_cv.data_pipeline import DataConfig, Food101DataModule
from food_cv.pipeline import MealPredictor
from food_cv.training import TrainConfig, train_classifier

paths = ProjectPaths.from_root(PROJECT_ROOT)
candidate_data_roots = [
    PROJECT_ROOT / "data",
    PROJECT_ROOT / "datasets",
    PROJECT_ROOT / "food101_data",
    Path("/content/data"),
    Path("/content/datasets"),
    PROJECT_ROOT,
]
data_root = next((p for p in candidate_data_roots if p.exists()), PROJECT_ROOT)
data_config = DataConfig(
    data_root=data_root,
    batch_size=16,
    num_workers=0,
    image_size=224,
    download_if_missing=True,
)

try:
    datamodule = Food101DataModule(data_config)
    train_loader, val_loader, test_loader = datamodule.build_dataloaders()
    class_names = datamodule.get_class_names()
    print(f"train={len(train_loader.dataset)} val={len(val_loader.dataset)} test={len(test_loader.dataset)}")
    print(f"labels={len(class_names)}")
except Exception as e:
    train_loader = val_loader = test_loader = None
    class_names = []
    print(f"Data loader 初始化失败: {e}")

model = build_resnet50_classifier(num_classes=101, freeze_backbone=True)
checkpoint_path = paths.model_dir / "food101_resnet50_fc_only.pt"

if train_loader is not None and val_loader is not None:
    metrics = train_classifier(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        save_path=checkpoint_path,
        config=TrainConfig(epochs=1, lr=1e-4, device="cuda", log_every_n_steps=20, max_steps_per_epoch=120),
    )
    print(json.dumps(metrics, ensure_ascii=False, indent=2))
else:
    print("跳过训练：请确认 Food-101 已下载并放到 data 目录")

demo_image = PROJECT_ROOT / "demo.jpg"
if checkpoint_path.exists() and demo_image.exists():
    predictor = MealPredictor(paths=paths, checkpoint_path=checkpoint_path, labels=class_names if class_names else None)
    result = predictor.predict_meal(demo_image)
    print(json.dumps(result, ensure_ascii=False, indent=2))
elif not checkpoint_path.exists():
    print("未找到训练后 checkpoint，已跳过推理。请先完成至少 1 个 epoch 训练。")
else:
    print("未找到 demo.jpg，推理演示已跳过")


ModuleNotFoundError: No module named 'torch'

In [6]:
from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive')

GITHUB_TOKEN = os.getenv('GITHUB_TOKEN', '')
REPO_OWNER = 'Blank4cc'
REPO_NAME = 'VisualDietician'
if not GITHUB_TOKEN:
    raise RuntimeError('请先在 Colab 环境变量中设置 GITHUB_TOKEN')
PRIVATE_REPO_URL = f'https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git'
DEST_PATH = f'/content/drive/MyDrive/{REPO_NAME}'

if not os.path.exists(DEST_PATH):
    print('正在尝试克隆私有仓库...')
    !git clone {PRIVATE_REPO_URL} {DEST_PATH}
else:
    print(f'目录已存在: {DEST_PATH}')

if os.path.exists(DEST_PATH):
    os.chdir(DEST_PATH)
    print(f'成功进入目录: {os.getcwd()}')
    !ls
else:
    print('克隆仍然失败，请检查 Token 权限或仓库路径是否正确。')
Path(DEST_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
正在尝试克隆私有仓库...
Cloning into '/content/drive/MyDrive/VisualDietician'...
remote: Enumerating objects: 75, done.
remote: Counting objects: 100% (75/75), done.
remote: Compressing objects: 100% (70/70), done.
remote: Total 75 (delta 12), reused 61 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (75/75), 18.03 MiB | 10.44 MiB/s, done.
Resolving deltas: 100% (12/12), done.
Updating files: 100% (57/57), done.
成功进入目录: /content/drive/MyDrive/VisualDietician
food_data  main.ipynb  src  workflow


In [7]:
import os

# 确保你在项目目录下
if os.path.exists(DEST_PATH):
    os.chdir(DEST_PATH)
    # 使用 git pull 拉取最新代码
    !git pull
    print("代码已尝试更新。")
else:
    print(f"未找到目录 {DEST_PATH}，请先运行克隆单元格。")

Already up to date.
代码已尝试更新。


In [8]:
import sys
from pathlib import Path

# 将当前路径（包含 src 目录的路径）加入 sys.path
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

print("当前 Python 路径已更新，现在可以尝试运行之前的代码单元格了。")

当前 Python 路径已更新，现在可以尝试运行之前的代码单元格了。
